# EarthForge — Vector Operations

This notebook demonstrates EarthForge's vector toolkit:
1. Vector info — inspect GeoParquet/Shapefile metadata without loading all features
2. Format conversion — Shapefile or GeoJSON → GeoParquet
3. Spatial query — filter features by bounding box with predicate pushdown
4. Format detection — identify unknown geospatial files

GeoParquet uses Apache Parquet as the storage layer — columnar, compressed, and queryable
with predicate pushdown. A spatial query over a 10M-feature dataset reads only the
row groups that intersect the bounding box, not the entire file.

In [ ]:
import sys

sys.path.insert(0, "../../packages/core/src")
sys.path.insert(0, "../../packages/vector/src")

from earthforge.core.config import EarthForgeProfile
from earthforge.core.formats import detect

# Default profile (no STAC operations in this notebook, but profile is required for the API)
profile = EarthForgeProfile(
    name="default",
    stac_api="https://earth-search.aws.element84.com/v1",
    storage_backend="local",
)

print("EarthForge vector toolkit ready")

## 1. Format Detection

`detect()` identifies geospatial file formats using a three-stage chain:
magic bytes → file extension → content inspection.

For GeoParquet, EarthForge reads the last 4096 bytes of the Parquet file and
looks for the `\x03geo` Thrift compact encoding of the `geo` metadata key — the
official GeoParquet 1.0 signal.

In [ ]:
import tempfile
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path

# Create a small test GeoParquet file
gdf = gpd.GeoDataFrame(
    {"name": ["Lexington", "Louisville", "Frankfort"], "pop": [322000, 633000, 28000]},
    geometry=[
        box(-84.6, 37.9, -84.4, 38.1),
        box(-85.9, 38.1, -85.6, 38.4),
        box(-84.9, 38.1, -84.8, 38.3),
    ],
    crs="EPSG:4326",
)

# Write to a temp GeoParquet file
tmpdir = Path(tempfile.mkdtemp())
geoparquet_path = tmpdir / "ky_cities.parquet"
gdf.to_parquet(geoparquet_path)

# Also create a plain Parquet (no geo metadata) for comparison
import pandas as pd

plain_parquet_path = tmpdir / "plain.parquet"
pd.DataFrame({"a": [1, 2, 3]}).to_parquet(plain_parquet_path)

# Detect both
geo_fmt = await detect(str(geoparquet_path))
plain_fmt = await detect(str(plain_parquet_path))

print(f"GeoParquet file → detected as: {geo_fmt}")
print(f"Plain Parquet   → detected as: {plain_fmt}")

## 2. Vector Info

`inspect_vector` reads the GeoParquet schema and geo metadata without loading
all features into memory. For large files (millions of features), this is
much faster than `geopandas.read_parquet()` followed by `.describe()`.

In [ ]:
from earthforge.vector.info import inspect_vector

info = await inspect_vector(str(geoparquet_path))

print(f"Format:       {info.format}")
print(f"Feature count:{info.feature_count}")
print(f"CRS:          {info.crs}")
print(f"Bounds:       {info.bounds}")
print(f"Geometry type:{info.geometry_type}")
print(f"Columns:      {info.columns}")

## 3. Format Conversion — Shapefile → GeoParquet

Convert any supported vector format to GeoParquet.
EarthForge uses geopandas for reading and writing; the output file is
GeoParquet 1.0 compliant with WKB geometry encoding.

In [ ]:
# First, write a Shapefile to convert from
shp_path = tmpdir / "ky_cities.shp"
gdf.to_file(shp_path)

print(f"Wrote Shapefile: {shp_path}")
print(f"Shapefile components: {list(tmpdir.glob('ky_cities.*'))}")

In [ ]:
from earthforge.vector.convert import convert_vector

output_path = tmpdir / "ky_cities_converted.parquet"

result = await convert_vector(
    source=str(shp_path),
    output=str(output_path),
    target_format="geoparquet",
)

print(f"Input:         {result.source_path}")
print(f"Output:        {result.output_path}")
print(f"Features:      {result.feature_count:,}")
print(f"Input size:    {result.source_size_bytes:,} bytes")
print(f"Output size:   {result.output_size_bytes:,} bytes")
print(f"Compression:   {result.source_size_bytes / result.output_size_bytes:.1f}x smaller")
print(f"Elapsed:       {result.elapsed_seconds:.3f}s")

## 4. Spatial Query with Bounding Box

`query_vector` filters features by bounding box using Parquet predicate pushdown.
Only row groups whose spatial metadata intersects the query bbox are read — the
rest of the file is skipped at the storage layer.

In [ ]:
from earthforge.vector.query import query_vector

# Query for features within the Lexington area only
lexington_bbox = (-84.7, 37.8, -84.3, 38.2)

query_result = await query_vector(
    source=str(output_path),
    bbox=lexington_bbox,
)

print(f"Query bbox:     {lexington_bbox}")
print(f"Features found: {query_result.feature_count}")
print(f"Elapsed:        {query_result.elapsed_seconds:.3f}s")
print()

# Show matching features
for _, row in query_result.geodataframe.iterrows():
    print(f"  {row['name']} (pop: {row['pop']:,})")

## 5. Real-World Data — Kentucky Wildlife Management Areas

The KY WMA dataset (from the ky_wma_demo.py script) demonstrates the workflow
on real state agency data. The script downloads and converts a Shapefile of
Kentucky's Wildlife Management Areas to GeoParquet, then queries by bbox.

Run the script directly for the full demo:
```bash
python examples/scripts/ky_wma_demo.py
```

Expected output:
- 83 WMA features covering 603,000+ acres
- Shapefile (5-file bundle) → GeoParquet (1 file, ~40% smaller)
- Spatial query over Jefferson County returns 2–4 features

## 6. Serialize to JSON

Vector query results are Pydantic models with a `geodataframe` field containing the
matched features. Use `.model_dump_json()` for the result metadata, and
GeoJSON for the geometry.

In [ ]:
import json

# GeoDataFrame → GeoJSON
if query_result.feature_count > 0:
    geojson = json.loads(query_result.geodataframe.to_json())
    print(json.dumps(geojson, indent=2))